**Linear SVM**

Install libraire

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix



In [ ]:
# chargement du dataset
ENTRAINEMENT = "regex"  # "" pour Topic Modeling, "regex" pour Regex

if ENTRAINEMENT == "regex":
    df = pd.read_csv("../data/labelled_topics/dataset_annotated_regex.csv")
else:
    df = pd.read_csv("../data/labelled_topics/dataset_avis.csv")

print(f"Dataset chargé : {len(df)} lignes (mode: {ENTRAINEMENT or 'topic_modeling'})")

In [ ]:
# colonnes cibles(multilabel)
labels = ["qualité produit", "service livraison", "service client"]


In [ ]:
# Données texte (features)
X = df["clean_comment"]



In [ ]:
# Données cibles
y = df[labels].values


In [ ]:
#TF-IDF vectorization

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

X_tfidf = tfidf.fit_transform(X)

In [ ]:
#Séparation Train / Test
# Nous séparons le dataset en ensembles d'entraînement et de test (80/20).
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Modèle SVM Multilabel
# nous entraînons un classifieur LinearSVC dans un schéma OneVsRest pour gérer les labels multiples.
# LinearSVC as base model
svm_model = LinearSVC(
    class_weight="balanced",
    max_iter=5000
)

# multilable avec OneVsRest
clf = OneVsRestClassifier(svm_model)

In [ ]:
#Validation croisée (F1 micro)
#Évaluation du modèle sur 5 folds avec le F1-score micro.
scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_tfidf,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro per fold :", cv_scores)
print("F1 micro average  :", cv_scores.mean())
print("Standard deviation :", cv_scores.std())

Entraînement et évaluation sur le test set

In [ ]:
# Train on train set
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)



# Scores per label
for i, col in enumerate(labels):
    acc = np.mean(y_test[:, i] == y_pred[:, i])
    f1 = f1_score(y_test[:, i], y_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# # Scores globaux
f1_micro = f1_score(y_test, y_pred, average="micro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"\nF1 micro    : {f1_micro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}")
#  Évaluation sur le jeu de test
print(classification_report(y_test, y_pred, target_names=labels))


In [ ]:
import joblib
from pathlib import Path

if ENTRAINEMENT == "regex":
    model_path = Path("../models/classification/svm/svm_regex.pkl")
else:
    model_path = Path("../models/classification/svm/svm_model.pkl")

model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, model_path)
print(f"Modèle sauvegardé dans {model_path}")

Matrices de confusion

In [ ]:

labels = ["qualité produit", "service livraison", "service client"]
for i, label in enumerate(labels):
    cm = confusion_matrix(y_test[:, i], y_pred[:, i])

    print(f"Confusion matrix for {label}:")
   # print(cm)
    cm_df = pd.DataFrame(cm,index = ["Vrai 0","Vrai 1"],columns = ["Prédit 0","Prédit 1"])
    print(cm_df)

Évaluation sur 100_avis_annote.csv

In [ ]:
#Évaluation sur 100_avis_annote.csv
df_avis = pd.read_csv("../data/test_dataset/100_avis_annote.csv", sep=";")
df_avis.head()

In [ ]:
# TF-IDF
X_avis_tfidf = tfidf.transform(df_avis['clean_comment'])
y_avis_true = df_avis[["qualité produit", "service livraison", "service client"]].values

In [ ]:
# Prédiction
y_avis_pred = clf.predict(X_avis_tfidf)

In [ ]:
# Scores per label
for i, col in enumerate(labels):
    acc = np.mean(y_avis_true[:, i] == y_avis_pred[:, i])
    f1 = f1_score(y_avis_true[:, i],y_avis_pred[:, i])
    print(f"Label '{col}': Accuracy = {acc:.3f}, F1-score = {f1:.3f}")

# Scores globaux
f1_micro = f1_score(y_avis_true, y_avis_pred, average="micro")
f1_weighted = f1_score(y_avis_true, y_avis_pred, average="weighted")

print(f"\nF1 micro    : {f1_micro:.4f}")
print(f"F1 weighted : {f1_weighted:.4f}")

In [ ]:
# Classification report
print("SVM Results on the 100 Manual Labels:")
print(classification_report(y_avis_true, y_avis_pred, target_names=labels))

In [ ]:
# Matrices de confusion

for i, label in enumerate(labels):
    cm = confusion_matrix(y_avis_true[:, i], y_avis_pred[:, i])

    cm_df = pd.DataFrame(
        cm,
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )

    print(f"\nMatrice de confusion — {label}")
    display(cm_df)